# 03 - Detect OCN And Publish Candidate Dataset

            This notebook loads generations, applies the lexical OCN detector, saves scored rows to Drive, publishes them to Hugging Face, and logs charts to W&B.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", "")
    if repo_url:
        target = Path("/content/empty-negations")
        if not target.exists():
            subprocess.run(["git", "clone", repo_url, str(target)], check=True)
        return target
    raise FileNotFoundError(
        "Could not find the empty-negations repo. Run this notebook from the repo, "
        "copy it to /content/drive/MyDrive/ocn_empty_negations, or set OCN_REPO_URL."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import json
            from pathlib import Path
            import matplotlib.pyplot as plt
            import pandas as pd
            import seaborn as sns
            import wandb
            from datasets import load_dataset

            from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, publish_dataframe_to_hf, save_dataframe
            from ocn.detectors import OCNDetector
            from ocn.metrics import detection_summary, grouped_ocn_rates, top_patterns

            paths = make_colab_paths()
            config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
            login_huggingface("HF_WRITE_ACCESS")
            run = login_wandb(project="ocn-empty-negations", name=f"detect-{config['run_id']}", config=config)
            sns.set_theme(style="whitegrid")

In [ ]:
generations = load_dataset(config["hf_generation_repo"], split="train").to_pandas()
            scored = OCNDetector().annotate_rows(generations, text_column="response")
            summary = detection_summary(scored)
            summary

In [ ]:
scored_path = save_dataframe(scored, Path(config["drive_data_root"]) / "ocn_detection.csv")
            repo_url = publish_dataframe_to_hf(
                scored,
                repo_id=config["hf_detection_repo"],
                split="train",
                private=config["hf_private"],
                card_path=REPO_ROOT / "dataset_cards/ocn_detection.md",
                commit_message=f"Publish OCN detection {config['run_id']}",
            )
            print("Saved:", scored_path)
            print("Published:", repo_url)

In [ ]:
model_rates = grouped_ocn_rates(scored, ["model_id", "model_stage", "decoding"])
            category_rates = grouped_ocn_rates(scored, ["category", "variant"])
            pattern_counts = top_patterns(scored, 20)

            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            sns.barplot(data=model_rates, y="model_id", x="ocn_rate", hue="decoding", ax=axes[0])
            axes[0].set_title("Candidate OCN rate by model")
            axes[0].set_xlim(0, 1)
            sns.barplot(data=category_rates.head(20), y="category", x="ocn_rate", hue="variant", ax=axes[1])
            axes[1].set_title("Top category/variant OCN rates")
            axes[1].set_xlim(0, 1)
            plt.tight_layout()
            fig_path = Path(config["drive_figure_root"]) / "03_ocn_rates.png"
            fig.savefig(fig_path, dpi=180, bbox_inches="tight")

            wandb.log({
                **summary.to_dict(),
                "model_rates": wandb.Table(dataframe=model_rates),
                "category_rates": wandb.Table(dataframe=category_rates),
                "top_patterns": wandb.Table(dataframe=pattern_counts),
                "ocn_rate_chart": wandb.Image(str(fig_path)),
            })
            run.finish()
            fig_path